In [2]:
import os
import json
import requests
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import faiss

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
print("API Key loaded:", OPENROUTER_API_KEY[:10], "...")

API Key loaded: sk-or-v1-5 ...


In [3]:
def load_documents(docs_folder="docs"):
    documents = []
    for filepath in Path(docs_folder).glob("*.md"):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        documents.append({
            "filename": filepath.name,
            "content": content
        })
        print(f"Loaded: {filepath.name} ({len(content)} chars)")
    return documents

documents = load_documents()
print(f"\nTotal documents loaded: {len(documents)}")

Loaded: doc1.md (3240 chars)
Loaded: doc2.md (4745 chars)
Loaded: doc3.md (3023 chars)

Total documents loaded: 3


In [5]:
def chunk_document(doc, chunk_size=100, overlap=20):
    chunks = []
    content = doc["content"]
    words = content.split()
    
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_text = " ".join(words[start:end])
        chunks.append({
            "filename": doc["filename"],
            "chunk_index": len(chunks),
            "text": chunk_text
        })
        start = end - overlap
    
    return chunks

# Chunk all documents
all_chunks = []
for doc in documents:
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)
    print(f"{doc['filename']}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

# Preview first chunk
print("\n--- Preview of first chunk ---")
print(all_chunks[0]["text"])

doc1.md: 6 chunks
doc2.md: 10 chunks
doc3.md: 6 chunks

Total chunks: 22

--- Preview of first chunk ---
# INDECIMAL — Company Overview & Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and AI Assistant Knowledge Base Last Updated: 2025-12-21 ## 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, and structured project tracking from inquiry to handover. ## 2) What Indecimal Promises (Customer-Facing Commitments) - Indecimal positions its offering as building “confidence” through commitment, not just contracts. - The approach emphasizes clarity and trust across the construction and ownership journey. ## 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. 2. Best


In [6]:
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]
print(f"Generating embeddings for {len(chunk_texts)} chunks...")

embeddings = embedder.encode(chunk_texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

print(f"\nEmbeddings shape: {embeddings.shape}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\soumy\project\mini_rag\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\soumy\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for 22 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings shape: (22, 384)


In [8]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
def retrieve_chunks(query, top_k=3):
    query_embedding = embedder.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "filename": all_chunks[idx]["filename"],
            "text": all_chunks[idx]["text"],
            "distance": distances[0][i]
        })
    return results

# Test it
query = "What are the construction packages and their prices?"
results = retrieve_chunks(query)

print(f"Query: {query}\n")
for r in results:
    print(f"Rank {r['rank']} | {r['filename']} | distance: {r['distance']:.4f}")
    print(r['text'][:200])
    print("---")
print(f"FAISS index built!")
print(f"Total vectors in index: {index.ntotal}")

Query: What are the construction packages and their prices?

Rank 1 | doc2.md | distance: 1.0579
or equivalent up to ₹400/bag ### Aggregates - 20mm & 40mm aggregates across all packages. ### Block Work - Solid concrete blocks 6" (external) & 4" (internal) - Pricing guidance shown as: - 6": up to 
---
Rank 2 | doc1.md | distance: 1.0772
## 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. 2. Best and Competitive Pricing - Fair pricing with no hidden charges. 
---
Rank 3 | doc1.md | distance: 1.1008
differentiators versus typical market alternatives: - Warranty & post-delivery support: long-term warranty/maintenance commitments. - Transparency: 100% transparent pricing and process. - Timelines: f
---
FAISS index built!
Total vectors in index: 22


In [15]:
def generate_answer(query, retrieved_chunks):
    # Build context from retrieved chunks
    context = ""
    for i, chunk in enumerate(retrieved_chunks):
        context += f"[Chunk {i+1} from {chunk['filename']}]\n{chunk['text']}\n\n"
    
    prompt = f"""You are a helpful assistant for Indecimal, a construction marketplace.
Answer the user's question using ONLY the context provided below.
If the answer is not found in the context, say "I don't have enough information to answer that."
Do not make up any information.

Context:
{context}

Question: {query}

Answer:"""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "google/gemma-3-4b-it:free",
            "messages": [{"role": "user", "content": prompt}]
        }
    )
    
    result = response.json()
    return result["choices"][0]["message"]["content"]

In [16]:
def rag_pipeline(query, top_k=3):
    print(f"Question: {query}")
    print("="*60)
    
    # Step 1: Retrieve
    chunks = retrieve_chunks(query, top_k=top_k)
    
    print("Retrieved Chunks:")
    for c in chunks:
        print(f"\n[Rank {c['rank']} | {c['filename']}]")
        print(c['text'][:300])
    
    print("\n" + "="*60)
    
    # Step 2: Generate
    print("Generating answer...\n")
    answer = generate_answer(query, chunks)
    
    print("Final Answer:")
    print(answer)
    return answer

# Test it!
rag_pipeline("What are the construction packages and their prices?")

Question: What are the construction packages and their prices?
Retrieved Chunks:

[Rank 1 | doc2.md]
or equivalent up to ₹400/bag ### Aggregates - 20mm & 40mm aggregates across all packages. ### Block Work - Solid concrete blocks 6" (external) & 4" (internal) - Pricing guidance shown as: - 6": up to ₹40 (+/- ₹3) per block - 4": up to ₹33 (+/- ₹3) per block ### RCC Mix - M20 or M25 (RCC design mix a

[Rank 2 | doc1.md]
## 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. 2. Best and Competitive Pricing - Fair pricing with no hidden charges. 3. Quality Assurance (445+ checks) - Strict quality control at every construction stage. 4. Stage-Ba

[Rank 3 | doc1.md]
differentiators versus typical market alternatives: - Warranty & post-delivery support: long-term warranty/maintenance commitments. - Transparency: 100% transparent pricing and process. - Timelines: fixed project timelines, with penalties for delays. - Qual

'The construction packages include aggregates, block work, RCC mix, and ceramic wall dado. \n\n*   Aggregates - 20mm & 40mm aggregates across all packages.\n*   Block Work - 6": up to ₹40 (+/- ₹3) per block, 4": up to ₹33 (+/- ₹3) per block.\n*   RCC Mix - M20 or M25.\n*   Ceramic Wall Dado - Essential: up to ₹40/sqft.'

In [14]:
# def generate_answer_debug(query, retrieved_chunks):
#     context = ""
#     for i, chunk in enumerate(retrieved_chunks):
#         context += f"[Chunk {i+1} from {chunk['filename']}]\n{chunk['text']}\n\n"
    
#     prompt = f"""You are a helpful assistant for Indecimal, a construction marketplace.
# Answer the user's question using ONLY the context provided below.
# If the answer is not found in the context, say "I don't have enough information to answer that."

# Context:
# {context}

# Question: {query}

# Answer:"""

#     response = requests.post(
#         "https://openrouter.ai/api/v1/chat/completions",
#         headers={
#             "Authorization": f"Bearer {OPENROUTER_API_KEY}",
#             "Content-Type": "application/json"
#         },
#         json={
#             "model": "google/gemma-3-4b-it:free",
#             "messages": [{"role": "user", "content": prompt}]
#         }
#     )
    
#     print("Status code:", response.status_code)
#     print("Raw response:")
#     print(json.dumps(response.json(), indent=2))

# # Test
# chunks = retrieve_chunks("What are the construction packages and their prices?")
# generate_answer_debug("What are the construction packages and their prices?", chunks)

Status code: 200
Raw response:
{
  "id": "gen-1774372029-GgQ6c1YjR11i9vmKrK6y",
  "object": "chat.completion",
  "created": 1774372029,
  "model": "google/gemma-3-4b-it:free",
  "provider": "Google AI Studio",
  "system_fingerprint": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "STOP",
      "message": {
        "role": "assistant",
        "content": "The construction packages include aggregates, block work, RCC mix, and ceramic wall dado. \n\n*   Aggregates - 20mm & 40mm aggregates across all packages.\n*   Block Work - 6\": up to \u20b940 (+/- \u20b93) per block, 4\": up to \u20b933 (+/- \u20b93) per block.\n*   RCC Mix - M20 or M25.\n*   Ceramic Wall Dado - Essential: up to \u20b940/sqft.",
        "refusal": null,
        "reasoning": null
      }
    }
  ],
  "usage": {
    "prompt_tokens": 569,
    "completion_tokens": 0,
    "total_tokens": 569,
    "cost": 0,
    "is_byok": false,
    "prompt_

In [22]:
rag_pipeline("What is Indecimal's payment and escrow system?")

Question: What is Indecimal's payment and escrow system?
Retrieved Chunks:

[Rank 1 | doc3.md]
# INDECIMAL — Customer Protection Policies, Quality System, and Guarantees (Internal Reference) Version: 1.0 Audience: Support, Ops, Project Management, AI Assistant Knowledge Base Last Updated: 2025-12-21 ## 1) Payment Safety & Stage Controls ### Escrow-Based Payment Model (Concept) - Customer paym

[Rank 2 | doc1.md]
# INDECIMAL — Company Overview & Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and AI Assistant Knowledge Base Last Updated: 2025-12-21 ## 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, 

[Rank 3 | doc1.md]
## 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. 2. Best and Competitive Pricing - Fair pricing with no hidden charges. 3. Quality Assurance (445+ checks) - Strict quality control at 

'Customer payments are made to an escrow account. A project manager verifies stage completion. Funds are disbursed to the construction partner after verification.'

In [19]:
rag_pipeline("What quality checks does Indecimal perform?");

Question: What quality checks does Indecimal perform?
Retrieved Chunks:

[Rank 1 | doc1.md]
## 3) What We Strive For (Operating Principles) 1. Smooth Construction Experience - Step-by-step support throughout the project. 2. Best and Competitive Pricing - Fair pricing with no hidden charges. 3. Quality Assurance (445+ checks) - Strict quality control at every construction stage. 4. Stage-Ba

[Rank 2 | doc1.md]
# INDECIMAL — Company Overview & Customer Journey (Internal Reference) Version: 1.0 Audience: Support, Sales, Product, and AI Assistant Knowledge Base Last Updated: 2025-12-21 ## 1) One-line Summary Indecimal provides end-to-end home construction support with transparent pricing, quality assurance, 

[Rank 3 | doc3.md]
Accountability ### Zero-Tolerance Policy on Construction Delays (Operational Mechanisms) Indecimal positions a system-driven approach to on-time delivery using: - Integrated project management system - Daily tracking of projects - Instant flagging of deviations - Au